# 01 — Data Audit

**Purpose:** Understand exactly what data we have before feature extraction.

**Questions answered:**
1. How many participants per class?
2. What's the train/val/test split?
3. Which modality files exist for each participant?
4. What are the data quality characteristics?
5. What threshold should we use for "high stress"?

In [ ]:
import sys
sys.path.insert(0, '..')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from src.data.cohort import build_cohort
from src.data.quality import check_file_availability, summarize_availability
from config import CLASS_NAMES_4, CLASS_NAMES_3, CLASS_NAMES_2
from config import LABEL_MAP_4_TO_3, LABEL_MAP_4_TO_2

pd.set_option('display.max_columns', None)
pd.set_option('display.width', 120)

## 1. Build Cohort

In [ ]:
cohort_df = build_cohort()
print(f'Cohort shape: {cohort_df.shape}')
print(f'Columns: {cohort_df.columns.tolist()}')

## 2. Class Distribution

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

# 4-class
counts_4 = cohort_df['label'].value_counts().sort_index()
axes[0].bar(CLASS_NAMES_4, counts_4.values, color=['#2ecc71', '#f39c12', '#e74c3c', '#8e44ad'])
axes[0].set_title('4-Class Distribution')
axes[0].set_ylabel('Count')
for i, v in enumerate(counts_4.values):
    axes[0].text(i, v + 10, str(v), ha='center', fontweight='bold')

# 3-class
labels_3 = cohort_df['label'].map(LABEL_MAP_4_TO_3)
counts_3 = labels_3.value_counts().sort_index()
axes[1].bar(CLASS_NAMES_3, counts_3.values, color=['#2ecc71', '#f39c12', '#e74c3c'])
axes[1].set_title('3-Class Distribution')
for i, v in enumerate(counts_3.values):
    axes[1].text(i, v + 10, str(v), ha='center', fontweight='bold')

# 2-class
labels_2 = cohort_df['label'].map(LABEL_MAP_4_TO_2)
counts_2 = labels_2.value_counts().sort_index()
axes[2].bar(CLASS_NAMES_2, counts_2.values, color=['#2ecc71', '#e74c3c'])
axes[2].set_title('2-Class Distribution')
for i, v in enumerate(counts_2.values):
    axes[2].text(i, v + 10, str(v), ha='center', fontweight='bold')

plt.tight_layout()
plt.savefig('../outputs/figures/class_distribution.png', dpi=150, bbox_inches='tight')
plt.show()

## 3. Train / Val / Test Split

In [ ]:
print('Dataset-provided 3-way split:\n')
split_table = pd.crosstab(
    cohort_df['recommended_split'],
    cohort_df['label']
)
split_table.columns = CLASS_NAMES_4
split_table['TOTAL'] = split_table.sum(axis=1)
print(split_table)
print(f'\nTotal: {len(cohort_df)}')

## 4. Modality File Availability

In [ ]:
availability = check_file_availability(cohort_df)
summarize_availability(availability)

In [ ]:
# SpO2 availability by class
merged_avail = availability.merge(cohort_df[['person_id', 'label']], on='person_id')

print('SpO2 availability by class:')
for label in range(4):
    sub = merged_avail[merged_avail['label'] == label]
    has = sub['spo2'].sum()
    total = len(sub)
    print(f'  {CLASS_NAMES_4[label]:>15s}: {has}/{total} ({has/total*100:.1f}%)')

## 5. Sensor Duration Distribution

In [ ]:
fig, ax = plt.subplots(figsize=(10, 4))
ax.hist(cohort_df['sensor_sampling_duration_days'], bins=50, edgecolor='black', alpha=0.7)
ax.axvline(x=14, color='red', linestyle='--', label='Min threshold (14 days)')
ax.set_xlabel('Sensor Duration (days)')
ax.set_ylabel('Count')
ax.set_title('Sensor Sampling Duration Distribution')
ax.legend()
plt.tight_layout()
plt.savefig('../outputs/figures/sensor_duration.png', dpi=150, bbox_inches='tight')
plt.show()

print(cohort_df['sensor_sampling_duration_days'].describe())

## 6. Key Findings Summary

### Cohort
- **1,655 participants** after >=14 day sensor filter
- 4 classes: healthy (580), prediabetes (427), oral_med (471), insulin (177)
- Insulin group is smallest at 10.7%

### Splits
- Dataset provides **3-way split**: train (1156), val (248), test (251)
- All classes represented in all splits

### Modality Availability
- **Heart Rate, Sleep, Activity, Calories**: 100% available
- **Respiratory Rate**: 99.6% (6 missing)
- **Stress**: 98.7% (22 missing)
- **SpO2**: 79.2% (344 missing) — the weakest modality
- SpO2 missingness is roughly uniform across classes (74-84%)
- 77.7% of participants have ALL 7 modalities

### Data Quality Findings
- Stress has THREE invalid values: -2 (sentinel), -1 (invalid reading, 44% of data!), and we keep >=0
- STRESS_HIGH_THRESHOLD set to 50 (top ~23% of valid readings)
- SPO2_LOW_THRESHOLD already set to 95 (clinical standard)

### Implications for Feature Engineering
- SpO2 features will be NaN for ~21% of participants — need imputation strategy
- Stress features must filter out -1 values (not just -2)
- All participants have at least 14 days of data — safe to use 14-day sequences